# 05 · Camada ouro
Une alertas e respostas numa tabela com o status do funil de cada alerta
e as métricas de tempo, pronta para o Power BI.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

In [ ]:
# Caminhos do projeto (relativos à pasta notebooks/)
bronze_dir = Path("../dados/bronze")
silver_dir = Path("../dados/prata")
gold_dir = Path("../dados/ouro")
source_file = bronze_dir / "Controle de alertas - CMF-SEAMA.xlsx"

# Data de referência: última atualização da planilha (cabeçalho da aba DADOS)
header = pd.read_excel(source_file, sheet_name="DADOS", nrows=0).columns
reference_date = next(pd.Timestamp(c) for c in header if isinstance(c, datetime))
print("Dados atualizados em", reference_date.date())

In [ ]:
df_alerts = pd.read_parquet(silver_dir / "alertas.parquet")
df_responses = pd.read_parquet(silver_dir / "respostas.parquet")

# Desfecho de cada linha de resposta, na ordem de precedência
row_conditions = [
    df_responses["houve_infracao"].eq("sim"),
    df_responses["houve_infracao"].eq("não"),
    df_responses["enviado_fiscalizacao"].eq("sim"),
    df_responses["motivo"].eq("falso positivo"),
    df_responses["motivo"].isin(["alerta tardio", "iuf tardio"]),
    df_responses["motivo"].eq("terra indígena"),
]
row_labels = ["fiscalizado com infração", "fiscalizado sem infração",
              "aguardando fiscalização", "falso positivo",
              "tardio (a definir)", "outro órgão"]

df_responses["status"] = np.select([c.fillna(False).astype(bool) for c in row_conditions],
                                   row_labels, default="triado sem desfecho")

# Por alerta, vale o desfecho mais avançado (menor posição na lista)
priority = {label: rank for rank, label in enumerate(row_labels + ["triado sem desfecho"])}
df_responses["prioridade"] = df_responses["status"].map(priority)
df_status = (df_responses.sort_values("prioridade")
             .drop_duplicates("id_alerta")[["id_alerta", "status"]])

# Primeira fiscalização de cada alerta
first_inspection = df_responses.groupby("id_alerta", as_index=False)["data_fiscalizacao"].min()

In [ ]:
# Separa o universo do funil: só alertas de supressão, publicados até a data da planilha da CMF
df_platform = df_alerts.copy()   # todos os alertas, para as análises de volume e tipos

is_suppression = df_platform["tipo_alerta"].str.startswith("Desmatamento")
date_for_cut = df_platform["data_publicacao"].fillna(df_platform["data_imagem_depois"])
in_period = date_for_cut <= reference_date
df_platform["no_funil"] = is_suppression & in_period

df_alerts = df_platform[df_platform["no_funil"]].drop(columns="no_funil").copy()
print(len(df_alerts), "alertas no funil |", len(df_platform), "alertas na plataforma")

# Nenhum alerta da planilha da CMF pode ficar fora do funil
cmf_outside = df_platform["na_planilha_cmf"] & ~df_platform["no_funil"]
print(cmf_outside.sum(), "alertas da CMF fora do funil (deve ser 0)")

In [ ]:
# Junta tudo na tabela de alertas, garantindo que nenhuma junção multiplique linhas
df_gold = (df_alerts
           .merge(df_status, on="id_alerta", how="left", validate="one_to_one")
           .merge(first_inspection, on="id_alerta", how="left", validate="one_to_one"))
df_gold["status"] = df_gold["status"].fillna("não triado")

# Alertas que têm resposta na planilha, mas só em linhas de quarentena
in_quarantine = df_gold["tem_resposta"] & df_gold["status"].eq("não triado")
df_gold.loc[in_quarantine, "status"] = "resposta em quarentena"

assert len(df_gold) == len(df_alerts), "A junção mudou o número de alertas"
print(df_gold["status"].value_counts().to_string())

In [ ]:
# Complementa com a plataforma: vale o status da CMF; sem desfecho na CMF, usa-se o da plataforma
df_sccon_status = pd.read_parquet(silver_dir / "status_sccon.parquet")
df_gold = df_gold.merge(df_sccon_status[["id_alerta", "status_sccon"]],
                        on="id_alerta", how="left", validate="one_to_one")

# Conferência antes da combinação: status da CMF × status da plataforma
print(pd.crosstab(df_gold["status"], df_gold["status_sccon"].fillna("(ausente)"),
                  margins=True).to_string())

from_platform = {"Encaminhado para fiscalização": "aguardando fiscalização",
                 "Falso positivo": "falso positivo",
                 "Fiscalizado com infração": "fiscalizado com infração",
                 "Fiscalizado sem infração": "fiscalizado sem infração"}
no_outcome = df_gold["status"].isin(["não triado", "resposta em quarentena", "triado sem desfecho"])
platform_status = df_gold["status_sccon"].map(from_platform)
use_platform = no_outcome & platform_status.notna()

df_gold.loc[use_platform, "status"] = platform_status[use_platform]
df_gold["fonte_status"] = np.where(use_platform, "plataforma SCCON",
                          np.where(df_gold["status"].eq("não triado"), "nenhuma", "planilha CMF"))

print("\n", use_platform.sum(), "alertas com status vindo da plataforma")
print(df_gold["status"].value_counts().to_string())

In [ ]:
# Renomeia o status que descreve ausência de registro, não ausência de triagem
df_gold["status"] = df_gold["status"].replace({"não triado": "sem registro de resposta"})

# Dias entre a publicação do alerta e a primeira fiscalização
df_gold["dias_ate_fiscalizacao"] = (df_gold["data_fiscalizacao"] - df_gold["data_publicacao"]).dt.days

# Datas inconsistentes: fiscalização antes da publicação ou depois da data da planilha
before_publication = df_gold["dias_ate_fiscalizacao"] < 0
after_reference = df_gold["data_fiscalizacao"] > reference_date
df_gold["data_inconsistente"] = before_publication | after_reference
df_gold.loc[df_gold["data_inconsistente"], "dias_ate_fiscalizacao"] = np.nan
print(before_publication.sum(), "fiscalizações antes da publicação |",
      after_reference.sum(), "fiscalizações depois da data da planilha")

# Coorte = mês de publicação; madura = já teve 90 dias para ser fiscalizada
df_gold["coorte"] = df_gold["data_publicacao"].dt.to_period("M").dt.to_timestamp()
df_gold["coorte_madura"] = (df_gold["data_publicacao"] + pd.Timedelta(days=90)) <= reference_date

# Marcadores do funil
discarded = ["falso positivo", "tardio (a definir)", "outro órgão"]
df_gold["descartado"] = df_gold["status"].isin(discarded)
df_gold["aguardando_fiscalizacao"] = df_gold["status"].eq("aguardando fiscalização")
df_gold["fiscalizado"] = df_gold["status"].str.startswith("fiscalizado")
df_gold["fiscalizado_90d"] = df_gold["dias_ate_fiscalizacao"].le(90)
df_gold["com_infracao"] = df_gold["status"].eq("fiscalizado com infração")

In [ ]:
# Regionalização do ES (Lei 9.768/2011, atualizada pela Lei 11.174/2020)
reference_dir = Path("../referencias")
df_regions = pd.read_csv(reference_dir / "regionalizacao_es.csv")

# Chave de junção: nome sem acentos, em minúsculas e sem espaços nas pontas
def name_key(series):
    return (series.str.normalize("NFKD").str.encode("ascii", "ignore")
                  .str.decode("ascii").str.lower().str.strip())

df_regions["chave_municipio"] = name_key(df_regions["nome_municipio"])
df_gold["chave_municipio"] = name_key(df_gold["municipio"])

# Municípios dos alertas sem correspondência na regionalização
unmatched = df_gold.loc[~df_gold["chave_municipio"].isin(df_regions["chave_municipio"]), "municipio"]
print(unmatched.nunique(), "nomes sem correspondência:", sorted(unmatched.unique()))

In [ ]:
# Leva o código do IBGE para os alertas (muitos alertas para um município)
df_gold = (df_gold
           .merge(df_regions[["chave_municipio", "cd_municipio"]],
                  on="chave_municipio", how="left", validate="many_to_one")
           .drop(columns="chave_municipio"))
df_gold["cd_municipio"] = df_gold["cd_municipio"].astype("Int64")

# Só os alertas do ES precisam ter município na regionalização
in_es = df_gold["uf"].eq("ES")
assert df_gold.loc[in_es, "cd_municipio"].notna().all(), "Há alerta do ES sem município na regionalização"

# Reconciliação com a Nota Técnica: municípios do ES com alerta que gerou infração
n_municipalities = df_gold.loc[df_gold["com_infracao"] & in_es, "cd_municipio"].nunique()
print(n_municipalities, "municípios do ES com alerta que gerou infração (NT: 69)")

# Tabela de municípios para o Power BI
gold_dir.mkdir(parents=True, exist_ok=True)
df_regions.drop(columns="chave_municipio").to_parquet(gold_dir / "municipios.parquet", index=False)

In [ ]:
# Tabela de todos os alertas da plataforma, com o código do município, para o Power BI
df_platform["chave_municipio"] = name_key(df_platform["municipio"])
df_platform = (df_platform
               .merge(df_regions[["chave_municipio", "cd_municipio"]],
                      on="chave_municipio", how="left", validate="many_to_one")
               .drop(columns="chave_municipio"))
df_platform["cd_municipio"] = df_platform["cd_municipio"].astype("Int64")

platform_columns = ["id_alerta", "tipo_alerta", "municipio", "uf", "cd_municipio", "area_ha",
                    "data_imagem_depois", "data_publicacao", "latitude", "longitude", "no_funil"]
df_platform[platform_columns].to_parquet(gold_dir / "alertas_plataforma.parquet", index=False)
print(len(df_platform), "alertas da plataforma salvos")

In [ ]:
# Checagens informativas
negative = df_gold["dias_ate_fiscalizacao"] < 0
print(negative.sum(), "alertas fiscalizados ANTES da publicação")
print(df_gold["coorte_madura"].sum(), "alertas em coortes maduras")
print("Mediana de dias até a fiscalização:", df_gold["dias_ate_fiscalizacao"].median())

# Salva na camada ouro
gold_dir.mkdir(parents=True, exist_ok=True)
df_gold.to_parquet(gold_dir / "alertas.parquet", index=False)
df_gold.info()

In [ ]:
# Tabelas de fiscalização para o modelo do Power BI
df_infractions = pd.read_parquet(silver_dir / "infracoes.parquet")
df_bridge = pd.read_parquet(silver_dir / "ponte_alerta_fiscalizacao.parquet")

# Fiscalizações: um número por linha, com a data
df_inspections = (pd.concat([df_bridge[["id_fiscalizacao"]], df_infractions[["id_fiscalizacao"]]])
                  .dropna().drop_duplicates().reset_index(drop=True))
inspection_dates = pd.read_parquet(silver_dir / "datas_fiscalizacao.parquet")
df_inspections = df_inspections.merge(inspection_dates, on="id_fiscalizacao",
                                      how="left", validate="one_to_one")

# Checagens
assert df_bridge["id_alerta"].isin(df_gold["id_alerta"]).all(), "Ponte aponta para alerta inexistente"
print(df_infractions["id_fiscalizacao"].isna().sum(), "infrações sem número de fiscalização")
print(len(df_inspections), "fiscalizações |", df_inspections["data_fiscalizacao"].isna().sum(), "sem data")

# Salva as três tabelas na camada ouro
df_inspections.to_parquet(gold_dir / "fiscalizacoes.parquet", index=False)
df_infractions.to_parquet(gold_dir / "infracoes.parquet", index=False)
df_bridge.to_parquet(gold_dir / "ponte_alerta_fiscalizacao.parquet", index=False)

In [ ]:
# Indicadores de qualidade do dado, recalculados a cada execução do pipeline
df_quarantine = pd.read_parquet(silver_dir / "quarentena_respostas.parquet")
inspected = df_responses["houve_infracao"].notna()
n_alerts = len(df_gold)
n_inspected = int(inspected.sum())

quality_rows = [
    # (indicador, ocorrências, base, efeito na análise, ação sugerida)
    ("Linhas da planilha de respostas em quarentena",
     len(df_quarantine), len(df_responses) + len(df_quarantine),
     "Excluídas da análise", "Corrigir o deslocamento das colunas na planilha"),
    ("Fiscalizações sem número de laudo",
     int((inspected & df_responses["id_fiscalizacao"].isna()).sum()), n_inspected,
     "Não se ligam a laudos nem a infrações", "Registrar o número do laudo"),
    ("Fiscalizações sem data",
     int((inspected & df_responses["data_fiscalizacao"].isna()).sum()), n_inspected,
     "Fora das métricas de tempo", "Registrar a data da fiscalização"),
    ("Fiscalizações anteriores à publicação do alerta",
     int(before_publication.sum()), int(df_gold["fiscalizado"].sum()),
     "Fora das métricas de tempo", "Verificar se vieram de denúncia ou se a data está errada"),
    ("Fiscalizações posteriores à data da planilha",
     int(after_reference.sum()), int(df_gold["fiscalizado"].sum()),
     "Fora das métricas de tempo", "Corrigir a data (provável erro de digitação)"),
    ("Alertas sem data de publicação",
     int(df_gold["data_publicacao"].isna().sum()), n_alerts,
     "Fora das métricas de tempo", "Completar a data de publicação"),
    ("Alertas sem unidade responsável",
     int(df_gold["unidade_responsavel"].isna().sum()), n_alerts,
     "Não se sabe quem atende o alerta", "Distribuir o alerta a uma unidade"),
    ("Status obtido só da plataforma SCCON",
     int(df_gold["fonte_status"].eq("plataforma SCCON").sum()), n_alerts,
     "A planilha não registra a etapa de envio a campo", "Registrar o envio na planilha ou na ferramenta"),
    ("Alertas sem nenhum registro de resposta",
     int(df_gold["status"].eq("sem registro de resposta").sum()), n_alerts,
     "Situação desconhecida", "Triar ou registrar a triagem"),
    ("Alertas de supressão fora da planilha da CMF",
     int((~df_gold["na_planilha_cmf"]).sum()), n_alerts,
     "Não entraram no controle de triagem", "Incluir na planilha ou na ferramenta"),
    ("Alertas com município principal fora do ES",
     int(df_gold["uf"].ne("ES").sum()), n_alerts,
     "Entram no funil, mas não nas vistas territoriais", "Nenhuma (regra do município principal)"),
]

quality_columns = ["indicador", "ocorrencias", "base", "efeito", "acao"]
df_quality = pd.DataFrame(quality_rows, columns=quality_columns)
df_quality["percentual"] = (df_quality["ocorrencias"] / df_quality["base"]).round(3)
df_quality["data_referencia"] = reference_date

# Tipo de cada indicador: erro de digitação ou lacuna no desenho do processo
indicator_type = {
    "Linhas da planilha de respostas em quarentena": "Erro de registro",
    "Fiscalizações sem número de laudo": "Erro de registro",
    "Fiscalizações sem data": "Erro de registro",
    "Fiscalizações anteriores à publicação do alerta": "Erro de registro",
    "Fiscalizações posteriores à data da planilha": "Erro de registro",
    "Alertas sem data de publicação": "Erro de registro",
    "Alertas sem unidade responsável": "Lacuna de processo",
    "Status obtido só da plataforma SCCON": "Lacuna de processo",
    "Alertas sem nenhum registro de resposta": "Lacuna de processo",
    "Alertas de supressão fora da planilha da CMF": "Lacuna de processo",
    "Alertas com município principal fora do ES": "Regra da análise",
}
df_quality["tipo"] = df_quality["indicador"].map(indicator_type)
assert df_quality["tipo"].notna().all(), "Há indicador sem tipo definido"

df_quality.to_parquet(gold_dir / "qualidade.parquet", index=False)
print(df_quality[["indicador", "ocorrencias", "base", "percentual"]].to_string())